# Stage 2 Nested CV Model Compare

这版 notebook 在 `aggregation v2` 的基础上，支持按运行时配置比较不同的
`variant × spec × model` 组合。

目标：
- 在统一 outer split 下比较 `variant × spec × model`
- 看非线性模型是否能把当前接近持平的 `P_taco` 信号放大
- 保持与 `modeling_validation_protocol.md` 一致的 coarse-grid 逻辑

说明：
- 这份 notebook 的实际比较范围由 runtime config 控制
- 主链可以保留更宽的 compare 范围
- robustness wrapper 可以把范围裁到单一 `variant × spec × model`

## Runtime Knob

这版 notebook 提供三个档位：
- `SEARCH_PROFILE = "smoke"`：默认，先做方向判断
- `SEARCH_PROFILE = "protocol"`：按 protocol 第一轮粗网格完整跑
- `SEARCH_PROFILE = "targeted_followup"`：在主链 winner 接近打平时，对更保守的正则/浅树配置做补充验证

推荐做法：
- 先跑 `smoke`
- 确认最有希望的 `variant × spec × model` 后，再切到 `protocol`
- 若 `macro_only` 与 `two-stage` 非常接近，再跑 `targeted_followup`

# Dense Export Sensitivity Stage 2 Final Runtime Config

这个 notebook 固定用于 `dense export sensitivity` 的 Stage 2 final compare：

- 输入固定读取 dense export sensitivity 的 `Stage 2` daily long 表
- `Stage 1 winner` 冻结继承主链：`linear_svm__sigmoid`
- outer 选择范围固定为：
  - models: `ridge`, `random_forest`, `xgboost_regressor`
  - spec: `two_stage_v2_core`
- holdout benchmark 固定为：
  - `random_walk_zero`
  - `macro_only_ridge`
  - `macro_only_random_forest`
- 输出目录固定到 `main_rerun/artifacts/dense_export_sensitivity/stage2/final_compare/`

In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "plan_v2.md").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)

DENSE_STAGE2_DIR = PROJECT_ROOT / "main_rerun" / "artifacts" / "dense_export_sensitivity" / "stage2"
DENSE_STAGE2_FINAL_DIR = DENSE_STAGE2_DIR / "final_compare"
DENSE_STAGE2_FINAL_DIR.mkdir(parents=True, exist_ok=True)

os.environ["STAGE2_HOLDOUT_LABEL"] = "primary_holdout"
os.environ["STAGE2_HOLDOUT_DISPLAY_TITLE"] = "Primary Holdout"
os.environ.setdefault("STAGE2_SEARCH_PROFILE", "smoke")
os.environ["STAGE2_COMPARE_DATA_PATH"] = str(
    DENSE_STAGE2_DIR / "stage2_daily_features_long_v2_dense_export_sensitivity.csv"
)
os.environ["STAGE2_COMPARE_RESULTS_DIR"] = str(DENSE_STAGE2_FINAL_DIR)
os.environ["STAGE2_ACTIVE_VARIANTS"] = "linear_svm__sigmoid"
os.environ["STAGE2_ACTIVE_MODELS"] = "ridge,random_forest,xgboost_regressor"
os.environ["STAGE2_SELECTION_MODEL_SPECS"] = "two_stage_v2_core"
os.environ["STAGE2_HOLDOUT_BENCHMARKS"] = "random_walk_zero,macro_only_ridge,macro_only_random_forest"

print("Project root:", PROJECT_ROOT)
print("Dense export Stage 2 feature dir:", DENSE_STAGE2_DIR)
print("Dense export Stage 2 final dir:", DENSE_STAGE2_FINAL_DIR)
print("Frozen Stage 1 variant:", "linear_svm__sigmoid")
print("STAGE2_SEARCH_PROFILE =", os.environ["STAGE2_SEARCH_PROFILE"])

In [ ]:
from pathlib import Path
import json
import os

os.environ["MPLCONFIGDIR"] = str(Path.cwd() / ".mpl-cache")

import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="talk")

DATA_PATH = Path("stage2/artifacts/stage2_daily_features_long_v2.csv")
SEARCH_PROFILE = os.environ.get("STAGE2_SEARCH_PROFILE", "smoke")
RESULTS_DIR = Path(f"stage2/artifacts/nested_cv_model_compare_{SEARCH_PROFILE}_v1")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COL = "gold_ret_1d"
DATE_COL = "date"
TRAIN_LABEL = "train_pool"
HOLDOUT_LABEL = os.environ.get("STAGE2_HOLDOUT_LABEL", "primary_holdout")
HOLDOUT_DISPLAY_TITLE = os.environ.get("STAGE2_HOLDOUT_DISPLAY_TITLE", "Primary Hold-out")
DATA_PATH = Path(os.environ.get("STAGE2_COMPARE_DATA_PATH", str(DATA_PATH)))
RESULTS_DIR = Path(os.environ.get("STAGE2_COMPARE_RESULTS_DIR", str(RESULTS_DIR)))
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

ACTIVE_VARIANTS = [
    key.strip()
    for key in os.environ.get("STAGE2_ACTIVE_VARIANTS", "").split(",")
    if key.strip()
]

MODEL_CANDIDATES = ["ridge", "random_forest", "xgboost_regressor"]
ACTIVE_MODELS = [
    key.strip()
    for key in os.environ.get("STAGE2_ACTIVE_MODELS", ",".join(MODEL_CANDIDATES)).split(",")
    if key.strip()
]
unknown_models = [key for key in ACTIVE_MODELS if key not in MODEL_CANDIDATES]
if unknown_models:
    raise ValueError(f"Unknown STAGE2_ACTIVE_MODELS entries: {unknown_models}")

OUTER_N_SPLITS = 5
OUTER_PURGE_DAYS = 2
OUTER_EMBARGO_DAYS = 5
OUTER_MIN_TRAIN_DAYS = 180
OUTER_MIN_VALID_DAYS = 30

INNER_N_SPLITS = 3
INNER_PURGE_DAYS = 2
INNER_EMBARGO_DAYS = 2
INNER_MIN_TRAIN_DAYS = 120
INNER_MIN_VALID_DAYS = 20

GAP_SPLIT_DAYS = 30
MIN_PREFIX_DAYS_LATER_SEGMENT = 30
RANDOM_STATE = 42

In [ ]:
P_TACO_V2_CORE_FEATURES = [
    "p_taco_any",
    "p_taco_top1",
    "p_taco_tail_010",
    "p_taco_any_decay_2d",
]

P_TACO_V2_INTERACTION_FEATURES = [
    "p_taco_any_x_vix_ma_5d_lag1",
    "p_taco_any_x_spx_drawdown_5d_lag1",
]

CORE_CONTROL_FEATURES = [
    "dxy_ret_1d_lag1",
    "real_yield_change_5d_lag1",
    "vix_ma_5d_lag1",
    "vix_change_5d_lag1",
    "spx_ret_1d_lag1",
    "spx_drawdown_5d_lag1",
    "gold_spx_corr_20d_lag1",
]

MODEL_SPEC_CATALOG = {
    "macro_only": CORE_CONTROL_FEATURES,
    "two_stage_v2_core": P_TACO_V2_CORE_FEATURES + CORE_CONTROL_FEATURES,
    "two_stage_v2_interact": P_TACO_V2_CORE_FEATURES + P_TACO_V2_INTERACTION_FEATURES + CORE_CONTROL_FEATURES,
}
SELECTION_MODEL_SPEC_NAMES = [
    key.strip()
    for key in os.environ.get(
        "STAGE2_SELECTION_MODEL_SPECS",
        "two_stage_v2_core",
    ).split(",")
    if key.strip()
]
unknown_selection_model_specs = [key for key in SELECTION_MODEL_SPEC_NAMES if key not in MODEL_SPEC_CATALOG]
if unknown_selection_model_specs:
    raise ValueError(f"Unknown STAGE2_SELECTION_MODEL_SPECS entries: {unknown_selection_model_specs}")
SELECTION_MODEL_SPECS = {key: MODEL_SPEC_CATALOG[key] for key in SELECTION_MODEL_SPEC_NAMES}

HOLDOUT_BENCHMARK_CATALOG = {
    "random_walk_zero": {
        "benchmark_label": "Random Walk (Zero)",
        "kind": "naive_zero",
    },
    "macro_only_ridge": {
        "benchmark_label": "Ridge + macro_only",
        "kind": "model",
        "model_name": "ridge",
        "spec_name": "macro_only",
    },
    "macro_only_random_forest": {
        "benchmark_label": "RandomForest + macro_only",
        "kind": "model",
        "model_name": "random_forest",
        "spec_name": "macro_only",
    },
}
HOLDOUT_BENCHMARK_KEYS = [
    key.strip()
    for key in os.environ.get(
        "STAGE2_HOLDOUT_BENCHMARKS",
        "random_walk_zero,macro_only_ridge,macro_only_random_forest",
    ).split(",")
    if key.strip()
]
unknown_holdout_benchmarks = [key for key in HOLDOUT_BENCHMARK_KEYS if key not in HOLDOUT_BENCHMARK_CATALOG]
if unknown_holdout_benchmarks:
    raise ValueError(f"Unknown STAGE2_HOLDOUT_BENCHMARKS entries: {unknown_holdout_benchmarks}")


def compute_regression_metrics(y_true, y_pred, benchmark_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    benchmark_pred = np.asarray(benchmark_pred, dtype=float)
    benchmark_mse = mean_squared_error(y_true, benchmark_pred)
    if benchmark_mse == 0:
        oos_r2 = np.nan
    else:
        oos_r2 = 1.0 - (mean_squared_error(y_true, y_pred) / benchmark_mse)
    return {
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
        "oos_r2": float(oos_r2) if pd.notna(oos_r2) else np.nan,
        "sign_accuracy": float((np.sign(y_pred) == np.sign(y_true)).mean()),
        "actual_mean": float(np.mean(y_true)),
        "pred_mean": float(np.mean(y_pred)),
        "benchmark_mean": float(np.mean(benchmark_pred)),
    }


def split_day_segments(unique_days, gap_days=30):
    unique_days = pd.Index(sorted(pd.Index(unique_days).unique()))
    if len(unique_days) == 0:
        return []

    segments = []
    start = 0
    diffs = np.diff(unique_days.view("int64")) / (24 * 3600 * 1e9)
    for idx, gap in enumerate(diffs, start=1):
        if gap > gap_days:
            segments.append(unique_days[start:idx])
            start = idx
    segments.append(unique_days[start:])
    return segments


def allocate_splits_to_segments(
    segments,
    n_splits,
    min_train_days,
    purge_days,
    min_valid_days,
    embargo_days,
    min_prefix_days_later_segment,
):
    eligible = []
    for seg_idx, segment in enumerate(segments):
        required_days = (
            min_train_days + purge_days + min_valid_days
            if seg_idx == 0
            else min_prefix_days_later_segment + purge_days + min_valid_days
        )
        if len(segment) < required_days:
            continue

        segment_capacity = int(
            (len(segment) - required_days) // max(min_valid_days + embargo_days, 1)
        ) + 1
        if segment_capacity <= 0:
            continue

        eligible.append(
            {
                "segment_idx": seg_idx,
                "segment_len": len(segment),
                "capacity": segment_capacity,
            }
        )

    if not eligible:
        raise ValueError("No eligible day segments available for the requested split design.")

    allocation = {item["segment_idx"]: 0 for item in eligible}
    remaining = n_splits

    for item in eligible:
        if remaining <= 0:
            break
        allocation[item["segment_idx"]] += 1
        remaining -= 1

    while remaining > 0:
        best_choice = None
        for item in eligible:
            current = allocation[item["segment_idx"]]
            if current >= item["capacity"]:
                continue
            score = item["segment_len"] / (current + 1)
            if best_choice is None or score > best_choice[0]:
                best_choice = (score, item["segment_idx"])

        if best_choice is None:
            break

        allocation[best_choice[1]] += 1
        remaining -= 1

    return allocation


def build_segmented_day_splits(
    unique_days,
    n_splits,
    purge_days,
    embargo_days,
    min_train_days,
    min_valid_days,
    gap_days=30,
    min_prefix_days_later_segment=30,
):
    unique_days = pd.Index(sorted(pd.Index(unique_days).unique()))
    segments = split_day_segments(unique_days, gap_days=gap_days)
    allocation = allocate_splits_to_segments(
        segments=segments,
        n_splits=n_splits,
        min_train_days=min_train_days,
        purge_days=purge_days,
        min_valid_days=min_valid_days,
        embargo_days=embargo_days,
        min_prefix_days_later_segment=min_prefix_days_later_segment,
    )

    splits = []
    prior_days = []
    fold_id = 1

    for seg_idx, segment in enumerate(segments):
        n_segment_splits = allocation.get(seg_idx, 0)
        if n_segment_splits == 0:
            prior_days.extend(list(segment))
            continue

        prefix_days = min_train_days if seg_idx == 0 else min_prefix_days_later_segment
        valid_start_min = prefix_days + purge_days
        available_end = len(segment) - min_valid_days
        if available_end < valid_start_min:
            prior_days.extend(list(segment))
            continue

        raw_starts = (
            np.array([valid_start_min])
            if n_segment_splits == 1
            else np.linspace(valid_start_min, available_end, n_segment_splits, dtype=int)
        )
        raw_starts = np.maximum.accumulate(raw_starts)
        previous_valid_end = None

        for local_idx, start_pos in enumerate(raw_starts, start=1):
            if previous_valid_end is not None:
                start_pos = max(start_pos, previous_valid_end + 1 + embargo_days)

            remaining_after = n_segment_splits - local_idx
            max_end_pos = (
                len(segment) - 1
                if remaining_after == 0
                else len(segment) - remaining_after * (min_valid_days + embargo_days) - 1
            )
            next_start = raw_starts[local_idx] if local_idx < n_segment_splits else len(segment)
            end_pos = max(
                start_pos + min_valid_days - 1,
                next_start - 1 - embargo_days,
            )
            end_pos = min(end_pos, max_end_pos)

            train_seg_end = start_pos - purge_days - 1
            train_days = pd.Index(list(prior_days) + list(segment[: train_seg_end + 1]))
            valid_days = segment[start_pos : end_pos + 1]

            purge_start = segment[train_seg_end + 1] if train_seg_end + 1 < len(segment) else pd.NaT
            purge_end = segment[start_pos - 1] if start_pos - 1 < len(segment) else pd.NaT
            embargo_start = segment[end_pos + 1] if end_pos + 1 < len(segment) else pd.NaT
            embargo_end_pos = min(end_pos + embargo_days, len(segment) - 1)
            embargo_end = segment[embargo_end_pos] if end_pos + 1 < len(segment) else pd.NaT

            splits.append(
                {
                    "fold_id": fold_id,
                    "segment_idx": seg_idx,
                    "train_days": train_days,
                    "valid_days": valid_days,
                    "train_start_day": train_days.min(),
                    "train_end_day": train_days.max(),
                    "valid_start_day": valid_days.min(),
                    "valid_end_day": valid_days.max(),
                    "purge_start_day": purge_start,
                    "purge_end_day": purge_end,
                    "embargo_start_day": embargo_start,
                    "embargo_end_day": embargo_end,
                    "train_day_count": len(train_days),
                    "valid_day_count": len(valid_days),
                }
            )
            previous_valid_end = end_pos
            fold_id += 1

        prior_days.extend(list(segment))

    return splits


def make_cv_index_pairs(frame, split_records):
    frame = frame.sort_values(DATE_COL).reset_index(drop=True)
    index_pairs = []
    split_meta_rows = []
    for split in split_records:
        train_idx = frame.index[frame[DATE_COL].isin(split["train_days"])].to_numpy()
        valid_idx = frame.index[frame[DATE_COL].isin(split["valid_days"])].to_numpy()
        if len(train_idx) == 0 or len(valid_idx) == 0:
            continue

        index_pairs.append((train_idx, valid_idx))
        split_meta_rows.append(
            {
                "fold_id": split["fold_id"],
                "segment_idx": split["segment_idx"],
                "train_start_day": split["train_start_day"],
                "train_end_day": split["train_end_day"],
                "valid_start_day": split["valid_start_day"],
                "valid_end_day": split["valid_end_day"],
                "purge_start_day": split["purge_start_day"],
                "purge_end_day": split["purge_end_day"],
                "embargo_start_day": split["embargo_start_day"],
                "embargo_end_day": split["embargo_end_day"],
                "train_day_count": split["train_day_count"],
                "valid_day_count": split["valid_day_count"],
                "train_rows": len(train_idx),
                "valid_rows": len(valid_idx),
            }
        )
    return index_pairs, pd.DataFrame(split_meta_rows)


def choose_inner_cv(frame):
    unique_days = pd.Index(sorted(frame[DATE_COL].dropna().unique()))
    try:
        split_records = build_segmented_day_splits(
            unique_days=unique_days,
            n_splits=INNER_N_SPLITS,
            purge_days=INNER_PURGE_DAYS,
            embargo_days=INNER_EMBARGO_DAYS,
            min_train_days=INNER_MIN_TRAIN_DAYS,
            min_valid_days=INNER_MIN_VALID_DAYS,
            gap_days=GAP_SPLIT_DAYS,
            min_prefix_days_later_segment=MIN_PREFIX_DAYS_LATER_SEGMENT,
        )
        cv_pairs, split_df = make_cv_index_pairs(frame, split_records)
        if len(cv_pairs) >= 2:
            return cv_pairs, split_df, "segmented_purged"
    except ValueError:
        pass

    fallback_splits = min(INNER_N_SPLITS, max(frame[DATE_COL].nunique() - 1, 1))
    if fallback_splits < 2:
        raise ValueError("Not enough unique dates to build inner CV.")

    fallback = TimeSeriesSplit(n_splits=fallback_splits)
    fallback_pairs = list(fallback.split(frame))
    fallback_rows = []
    for idx, (train_idx, valid_idx) in enumerate(fallback_pairs, start=1):
        fallback_rows.append(
            {
                "fold_id": idx,
                "segment_idx": -1,
                "train_start_day": frame.iloc[train_idx][DATE_COL].min(),
                "train_end_day": frame.iloc[train_idx][DATE_COL].max(),
                "valid_start_day": frame.iloc[valid_idx][DATE_COL].min(),
                "valid_end_day": frame.iloc[valid_idx][DATE_COL].max(),
                "purge_start_day": pd.NaT,
                "purge_end_day": pd.NaT,
                "embargo_start_day": pd.NaT,
                "embargo_end_day": pd.NaT,
                "train_day_count": frame.iloc[train_idx][DATE_COL].nunique(),
                "valid_day_count": frame.iloc[valid_idx][DATE_COL].nunique(),
                "train_rows": len(train_idx),
                "valid_rows": len(valid_idx),
            }
        )
    return fallback_pairs, pd.DataFrame(fallback_rows), "time_series_fallback"


def build_model_bundle(model_name, search_profile):
    if model_name == "ridge":
        estimator = Ridge()
        if search_profile == "protocol":
            grid = {"model__alpha": [0.1, 1.0, 10.0]}
        elif search_profile == "targeted_followup":
            grid = {"model__alpha": [10.0, 30.0, 100.0, 300.0]}
        else:
            grid = {"model__alpha": [0.1, 1.0, 10.0, 100.0]}
    elif model_name == "random_forest":
        estimator = RandomForestRegressor(
            random_state=RANDOM_STATE,
            n_jobs=1,
        )
        if search_profile == "protocol":
            grid = {
                "model__n_estimators": [200],
                "model__max_depth": [3, 5],
                "model__min_samples_leaf": [1, 5],
                "model__max_features": ["sqrt", 0.5],
            }
        elif search_profile == "targeted_followup":
            grid = {
                "model__n_estimators": [200, 500],
                "model__max_depth": [2, 3, 5],
                "model__min_samples_leaf": [1, 5],
                "model__max_features": ["sqrt", 0.5],
            }
        else:
            grid = {
                "model__n_estimators": [200],
                "model__max_depth": [3, 5],
                "model__min_samples_leaf": [1, 5],
                "model__max_features": ["sqrt", 0.5],
            }
    elif model_name == "xgboost_regressor":
        estimator = XGBRegressor(
            objective="reg:squarederror",
            eval_metric="rmse",
            random_state=RANDOM_STATE,
            n_jobs=1,
            tree_method="hist",
            verbosity=0,
        )
        if search_profile == "protocol":
            grid = {
                "model__n_estimators": [200],
                "model__max_depth": [2, 3],
                "model__learning_rate": [0.03, 0.08],
                "model__min_child_weight": [1, 5],
                "model__subsample": [0.8],
                "model__colsample_bytree": [0.8],
            }
        elif search_profile == "targeted_followup":
            grid = {
                "model__n_estimators": [200, 500],
                "model__max_depth": [1, 2, 3],
                "model__learning_rate": [0.01, 0.03],
                "model__min_child_weight": [5, 10],
                "model__subsample": [0.8],
                "model__colsample_bytree": [0.8],
            }
        else:
            grid = {
                "model__n_estimators": [200],
                "model__max_depth": [2, 3],
                "model__learning_rate": [0.03, 0.08],
                "model__min_child_weight": [1, 5],
                "model__subsample": [0.8],
                "model__colsample_bytree": [0.8],
            }
    else:
        raise ValueError(f"Unknown model_name: {model_name}")

    return estimator, grid


def make_model_pipeline(features, model_name):
    estimator, grid = build_model_bundle(model_name=model_name, search_profile=SEARCH_PROFILE)
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                features,
            ),
        ],
        remainder="drop",
    )
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", estimator),
        ]
    )
    return pipeline, grid


def extract_model_importance(estimator, features, variant_key, spec_name, model_name):
    fitted_model = estimator.named_steps["model"]
    if hasattr(fitted_model, "coef_"):
        values = fitted_model.coef_
        importance_type = "coefficient"
    elif hasattr(fitted_model, "feature_importances_"):
        values = fitted_model.feature_importances_
        importance_type = "feature_importance"
    else:
        values = np.repeat(np.nan, len(features))
        importance_type = "not_available"

    return pd.DataFrame(
        {
            "variant_key": variant_key,
            "model_name": model_name,
            "spec_name": spec_name,
            "feature_name": features,
            "importance_value": values,
            "abs_importance_value": np.abs(values),
            "importance_type": importance_type,
        }
    )


def flatten_summary_columns(frame):
    flat_cols = []
    for col in frame.columns:
        if isinstance(col, tuple):
            flat_cols.append("_".join([str(part) for part in col if part]).strip("_"))
        else:
            flat_cols.append(str(col))
    out = frame.copy()
    out.columns = flat_cols
    return out.reset_index()

In [ ]:
df = pd.read_csv(DATA_PATH)
df[DATE_COL] = pd.to_datetime(df[DATE_COL]).dt.normalize()
df = df.sort_values(["variant_key", DATE_COL]).reset_index(drop=True)

if not ACTIVE_VARIANTS:
    ACTIVE_VARIANTS = sorted(df["variant_key"].dropna().unique().tolist())
missing_variants = [key for key in ACTIVE_VARIANTS if key not in df["variant_key"].dropna().unique()]
if missing_variants:
    raise ValueError(f"Missing active variants in compare input: {missing_variants}")

active_feature_columns = sorted(
    set(
        feature
        for feature_cols in list(SELECTION_MODEL_SPECS.values()) + [MODEL_SPEC_CATALOG["macro_only"]]
        for feature in feature_cols
    )
)
required_columns = sorted(
    set([TARGET_COL, DATE_COL, "stage2_split", "variant_key"] + active_feature_columns)
)
missing_required = [col for col in required_columns if col not in df.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

main_df = df[df["stage2_split"].isin([TRAIN_LABEL, HOLDOUT_LABEL])].copy()
main_df = main_df.dropna(subset=[TARGET_COL]).sort_values(["variant_key", DATE_COL]).reset_index(drop=True)

print(f"Loaded {len(df):,} rows and {df.shape[1]} columns from {DATA_PATH}")
print(f"Main sample rows with non-null target: {len(main_df):,}")
print("Search profile:", SEARCH_PROFILE)

In [ ]:
split_summary = (
    main_df.groupby(["variant_key", "stage2_split"])
    .agg(
        n_rows=(DATE_COL, "size"),
        n_unique_dates=(DATE_COL, "nunique"),
        first_date=(DATE_COL, "min"),
        last_date=(DATE_COL, "max"),
        target_mean=(TARGET_COL, "mean"),
        target_std=(TARGET_COL, "std"),
    )
    .reset_index()
    .sort_values(["variant_key", "stage2_split"])
)
display(split_summary)

missing_summary = (
    main_df[CORE_CONTROL_FEATURES + P_TACO_V2_CORE_FEATURES + P_TACO_V2_INTERACTION_FEATURES + [TARGET_COL]]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .rename("missing_rate")
    .to_frame()
)
display(missing_summary)

In [ ]:
outer_unique_days = pd.Index(
    sorted(main_df.loc[main_df["stage2_split"] == TRAIN_LABEL, DATE_COL].dropna().unique())
)
outer_split_records = build_segmented_day_splits(
    unique_days=outer_unique_days,
    n_splits=OUTER_N_SPLITS,
    purge_days=OUTER_PURGE_DAYS,
    embargo_days=OUTER_EMBARGO_DAYS,
    min_train_days=OUTER_MIN_TRAIN_DAYS,
    min_valid_days=OUTER_MIN_VALID_DAYS,
    gap_days=GAP_SPLIT_DAYS,
    min_prefix_days_later_segment=MIN_PREFIX_DAYS_LATER_SEGMENT,
)
_, outer_splits_df = make_cv_index_pairs(
    frame=main_df[main_df["variant_key"] == ACTIVE_VARIANTS[0]].copy(),
    split_records=outer_split_records,
)
display(outer_splits_df)

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))
for _, row in outer_splits_df.iterrows():
    ax.plot([row["train_start_day"], row["train_end_day"]], [row["fold_id"], row["fold_id"]], color="#457b9d", linewidth=6, solid_capstyle="butt")
    ax.plot([row["valid_start_day"], row["valid_end_day"]], [row["fold_id"], row["fold_id"]], color="#e76f51", linewidth=6, solid_capstyle="butt")
ax.set_title("Stage 2 Outer Splits: Train (blue) vs Validation (red)")
ax.set_xlabel("date")
ax.set_ylabel("fold_id")
plt.tight_layout()
plt.show()

In [ ]:
outer_fold_metric_rows = []
outer_prediction_frames = []
best_param_rows = []
holdout_metric_rows = []
holdout_prediction_frames = []
holdout_importance_frames = []
holdout_benchmark_rows = []
holdout_benchmark_prediction_frames = []

for variant_key in ACTIVE_VARIANTS:
    variant_df = (
        main_df[main_df["variant_key"] == variant_key]
        .sort_values(DATE_COL)
        .reset_index(drop=True)
    )
    train_pool_df = variant_df[variant_df["stage2_split"] == TRAIN_LABEL].copy().reset_index(drop=True)
    holdout_df = variant_df[variant_df["stage2_split"] == HOLDOUT_LABEL].copy().reset_index(drop=True)

    print(
        f"{variant_key}: train_pool={len(train_pool_df):,} rows / {train_pool_df[DATE_COL].nunique():,} days, "
        f"holdout={len(holdout_df):,} rows / {holdout_df[DATE_COL].nunique():,} days"
    )

    for model_name in ACTIVE_MODELS:
        print(f"  - model {model_name}")

        for spec_name, feature_cols in SELECTION_MODEL_SPECS.items():
            print(f"    * spec {spec_name}")

            for outer_split in outer_split_records:
                outer_train = train_pool_df[train_pool_df[DATE_COL].isin(outer_split["train_days"])].copy().reset_index(drop=True)
                outer_valid = train_pool_df[train_pool_df[DATE_COL].isin(outer_split["valid_days"])].copy().reset_index(drop=True)

                inner_cv_pairs, inner_splits_df, inner_strategy = choose_inner_cv(outer_train)
                pipeline, param_grid = make_model_pipeline(feature_cols, model_name=model_name)
                search = GridSearchCV(
                    estimator=pipeline,
                    param_grid=param_grid,
                    scoring="neg_root_mean_squared_error",
                    cv=inner_cv_pairs,
                    refit=True,
                    n_jobs=None,
                )
                search.fit(outer_train[feature_cols], outer_train[TARGET_COL])

                valid_pred = search.best_estimator_.predict(outer_valid[feature_cols])
                outer_benchmark_pred = np.repeat(
                    outer_train[TARGET_COL].mean(),
                    len(outer_valid),
                )
                fold_metrics = compute_regression_metrics(
                    outer_valid[TARGET_COL].to_numpy(),
                    valid_pred,
                    benchmark_pred=outer_benchmark_pred,
                )
                fold_metrics.update(
                    {
                        "variant_key": variant_key,
                        "model_name": model_name,
                        "spec_name": spec_name,
                        "fold_id": outer_split["fold_id"],
                        "segment_idx": outer_split["segment_idx"],
                        "inner_cv_strategy": inner_strategy,
                        "best_params_json": json.dumps(search.best_params_, ensure_ascii=False),
                        "best_inner_rmse": float(-search.best_score_),
                        "train_start_day": outer_split["train_start_day"],
                        "train_end_day": outer_split["train_end_day"],
                        "valid_start_day": outer_split["valid_start_day"],
                        "valid_end_day": outer_split["valid_end_day"],
                        "train_day_count": outer_split["train_day_count"],
                        "valid_day_count": outer_split["valid_day_count"],
                        "train_rows": len(outer_train),
                        "valid_rows": len(outer_valid),
                    }
                )
                outer_fold_metric_rows.append(fold_metrics)

                outer_prediction_frames.append(
                    pd.DataFrame(
                        {
                            "date": outer_valid[DATE_COL].to_numpy(),
                            "variant_key": variant_key,
                            "model_name": model_name,
                            "spec_name": spec_name,
                            "fold_id": outer_split["fold_id"],
                            "y_true": outer_valid[TARGET_COL].to_numpy(),
                            "y_pred": valid_pred,
                        }
                    )
                )

                best_param_rows.append(
                    {
                        "variant_key": variant_key,
                        "model_name": model_name,
                        "spec_name": spec_name,
                        "fold_id": outer_split["fold_id"],
                        "segment_idx": outer_split["segment_idx"],
                        "best_params_json": json.dumps(search.best_params_, ensure_ascii=False),
                        "best_inner_rmse": float(-search.best_score_),
                        "inner_cv_strategy": inner_strategy,
                        "feature_list_json": json.dumps(feature_cols, ensure_ascii=False),
                    }
                )

            full_inner_pairs, full_inner_splits_df, full_inner_strategy = choose_inner_cv(train_pool_df)
            full_pipeline, full_param_grid = make_model_pipeline(feature_cols, model_name=model_name)
            full_search = GridSearchCV(
                estimator=full_pipeline,
                param_grid=full_param_grid,
                scoring="neg_root_mean_squared_error",
                cv=full_inner_pairs,
                refit=True,
                n_jobs=None,
            )
            full_search.fit(train_pool_df[feature_cols], train_pool_df[TARGET_COL])
            holdout_pred = full_search.best_estimator_.predict(holdout_df[feature_cols])
            holdout_benchmark_pred = np.repeat(
                train_pool_df[TARGET_COL].mean(),
                len(holdout_df),
            )
            holdout_metrics = compute_regression_metrics(
                holdout_df[TARGET_COL].to_numpy(),
                holdout_pred,
                benchmark_pred=holdout_benchmark_pred,
            )
            holdout_metrics.update(
                {
                    "variant_key": variant_key,
                    "model_name": model_name,
                    "spec_name": spec_name,
                    "best_params_json": json.dumps(full_search.best_params_, ensure_ascii=False),
                    "best_inner_rmse": float(-full_search.best_score_),
                    "inner_cv_strategy": full_inner_strategy,
                    "n_obs": len(holdout_df),
                }
            )
            holdout_metric_rows.append(holdout_metrics)

            holdout_prediction_frames.append(
                pd.DataFrame(
                    {
                        "date": holdout_df[DATE_COL].to_numpy(),
                        "variant_key": variant_key,
                        "model_name": model_name,
                        "spec_name": spec_name,
                        "y_true": holdout_df[TARGET_COL].to_numpy(),
                        "y_pred": holdout_pred,
                    }
                )
            )

            holdout_importance_frames.append(
                extract_model_importance(
                    estimator=full_search.best_estimator_,
                    features=feature_cols,
                    variant_key=variant_key,
                    spec_name=spec_name,
                    model_name=model_name,
                )
            )

    train_pool_benchmark_pred = np.repeat(
        train_pool_df[TARGET_COL].mean(),
        len(holdout_df),
    )
    for benchmark_key in HOLDOUT_BENCHMARK_KEYS:
        benchmark_config = HOLDOUT_BENCHMARK_CATALOG[benchmark_key]
        if benchmark_config["kind"] == "naive_zero":
            holdout_pred = np.zeros(len(holdout_df))
            holdout_metrics = compute_regression_metrics(
                holdout_df[TARGET_COL].to_numpy(),
                holdout_pred,
                benchmark_pred=train_pool_benchmark_pred,
            )
            holdout_metrics.update(
                {
                    "variant_key": variant_key,
                    "benchmark_key": benchmark_key,
                    "benchmark_label": benchmark_config["benchmark_label"],
                    "model_name": "benchmark",
                    "spec_name": benchmark_key,
                    "reference_model_name": np.nan,
                    "reference_spec_name": np.nan,
                    "best_params_json": "",
                    "best_inner_rmse": np.nan,
                    "inner_cv_strategy": "naive_zero",
                    "n_obs": len(holdout_df),
                }
            )
            holdout_benchmark_rows.append(holdout_metrics)
            holdout_benchmark_prediction_frames.append(
                pd.DataFrame(
                    {
                        "date": holdout_df[DATE_COL].to_numpy(),
                        "variant_key": variant_key,
                        "benchmark_key": benchmark_key,
                        "benchmark_label": benchmark_config["benchmark_label"],
                        "y_true": holdout_df[TARGET_COL].to_numpy(),
                        "y_pred": holdout_pred,
                    }
                )
            )
            continue

        benchmark_model_name = benchmark_config["model_name"]
        benchmark_spec_name = benchmark_config["spec_name"]
        benchmark_feature_cols = MODEL_SPEC_CATALOG[benchmark_spec_name]
        full_inner_pairs, _, full_inner_strategy = choose_inner_cv(train_pool_df)
        full_pipeline, full_param_grid = make_model_pipeline(
            benchmark_feature_cols,
            model_name=benchmark_model_name,
        )
        full_search = GridSearchCV(
            estimator=full_pipeline,
            param_grid=full_param_grid,
            scoring="neg_root_mean_squared_error",
            cv=full_inner_pairs,
            refit=True,
            n_jobs=None,
        )
        full_search.fit(train_pool_df[benchmark_feature_cols], train_pool_df[TARGET_COL])
        holdout_pred = full_search.best_estimator_.predict(holdout_df[benchmark_feature_cols])
        holdout_metrics = compute_regression_metrics(
            holdout_df[TARGET_COL].to_numpy(),
            holdout_pred,
            benchmark_pred=train_pool_benchmark_pred,
        )
        holdout_metrics.update(
            {
                "variant_key": variant_key,
                "benchmark_key": benchmark_key,
                "benchmark_label": benchmark_config["benchmark_label"],
                "model_name": "benchmark",
                "spec_name": benchmark_key,
                "reference_model_name": benchmark_model_name,
                "reference_spec_name": benchmark_spec_name,
                "best_params_json": json.dumps(full_search.best_params_, ensure_ascii=False),
                "best_inner_rmse": float(-full_search.best_score_),
                "inner_cv_strategy": full_inner_strategy,
                "n_obs": len(holdout_df),
            }
        )
        holdout_benchmark_rows.append(holdout_metrics)
        holdout_benchmark_prediction_frames.append(
            pd.DataFrame(
                {
                    "date": holdout_df[DATE_COL].to_numpy(),
                    "variant_key": variant_key,
                    "benchmark_key": benchmark_key,
                    "benchmark_label": benchmark_config["benchmark_label"],
                    "y_true": holdout_df[TARGET_COL].to_numpy(),
                    "y_pred": holdout_pred,
                }
            )
        )

outer_fold_metrics_df = pd.DataFrame(outer_fold_metric_rows)
outer_predictions_df = pd.concat(outer_prediction_frames, ignore_index=True)
best_params_df = pd.DataFrame(best_param_rows)
holdout_metrics_df = pd.DataFrame(holdout_metric_rows).sort_values(["variant_key", "model_name", "spec_name"]).reset_index(drop=True)
holdout_predictions_df = pd.concat(holdout_prediction_frames, ignore_index=True)
holdout_importances_df = pd.concat(holdout_importance_frames, ignore_index=True)
holdout_benchmark_metrics_df = pd.DataFrame(holdout_benchmark_rows).sort_values(["variant_key", "benchmark_key"]).reset_index(drop=True)
holdout_benchmark_predictions_df = pd.concat(holdout_benchmark_prediction_frames, ignore_index=True)

outer_summary_df = flatten_summary_columns(
    outer_fold_metrics_df.groupby(["variant_key", "model_name", "spec_name"])[["rmse", "mae", "r2", "oos_r2", "sign_accuracy"]]
    .agg(["mean", "std"])
)

selected_model_rows = []
for variant_key in ACTIVE_VARIANTS:
    variant_outer = outer_summary_df[outer_summary_df["variant_key"] == variant_key].copy()
    selected_row = (
        variant_outer.sort_values(
            ["rmse_mean", "mae_mean", "oos_r2_mean"],
            ascending=[True, True, False],
        )
        .iloc[0]
        .to_dict()
    )
    selected_model_rows.append(selected_row)
selected_model_df = pd.DataFrame(selected_model_rows)

display(outer_summary_df.sort_values(["variant_key", "model_name", "rmse_mean"]))
display(selected_model_df[["variant_key", "model_name", "spec_name", "rmse_mean", "mae_mean", "oos_r2_mean", "sign_accuracy_mean"]])
display(holdout_metrics_df[["variant_key", "model_name", "spec_name", "rmse", "mae", "r2", "oos_r2", "sign_accuracy"]])
display(holdout_benchmark_metrics_df[["variant_key", "benchmark_label", "rmse", "mae", "r2", "oos_r2", "sign_accuracy"]])

In [ ]:
holdout_compare_rows = []
for _, selected_row in selected_model_df.iterrows():
    variant_key = selected_row["variant_key"]
    selected_holdout_row = holdout_metrics_df[
        (holdout_metrics_df["variant_key"] == variant_key)
        & (holdout_metrics_df["model_name"] == selected_row["model_name"])
        & (holdout_metrics_df["spec_name"] == selected_row["spec_name"])
    ].iloc[0]

    holdout_compare_rows.append(
        {
            "variant_key": variant_key,
            "benchmark_key": "selected_final_model",
            "benchmark_label": f'{selected_row["model_name"]} + {selected_row["spec_name"]}',
            "rmse": float(selected_holdout_row["rmse"]),
            "mae": float(selected_holdout_row["mae"]),
            "r2": float(selected_holdout_row["r2"]),
            "oos_r2": float(selected_holdout_row["oos_r2"]),
            "sign_accuracy": float(selected_holdout_row["sign_accuracy"]),
            "delta_rmse_vs_selected": 0.0,
            "delta_mae_vs_selected": 0.0,
            "delta_r2_vs_selected": 0.0,
            "delta_oos_r2_vs_selected": 0.0,
            "delta_sign_acc_vs_selected": 0.0,
        }
    )

    for _, benchmark_row in holdout_benchmark_metrics_df[
        holdout_benchmark_metrics_df["variant_key"] == variant_key
    ].iterrows():
        holdout_compare_rows.append(
            {
                "variant_key": variant_key,
                "benchmark_key": benchmark_row["benchmark_key"],
                "benchmark_label": benchmark_row["benchmark_label"],
                "rmse": float(benchmark_row["rmse"]),
                "mae": float(benchmark_row["mae"]),
                "r2": float(benchmark_row["r2"]),
                "oos_r2": float(benchmark_row["oos_r2"]),
                "sign_accuracy": float(benchmark_row["sign_accuracy"]),
                "delta_rmse_vs_selected": float(benchmark_row["rmse"] - selected_holdout_row["rmse"]),
                "delta_mae_vs_selected": float(benchmark_row["mae"] - selected_holdout_row["mae"]),
                "delta_r2_vs_selected": float(benchmark_row["r2"] - selected_holdout_row["r2"]),
                "delta_oos_r2_vs_selected": float(benchmark_row["oos_r2"] - selected_holdout_row["oos_r2"]),
                "delta_sign_acc_vs_selected": float(benchmark_row["sign_accuracy"] - selected_holdout_row["sign_accuracy"]),
            }
        )
holdout_compare = pd.DataFrame(holdout_compare_rows).sort_values(["variant_key", "delta_rmse_vs_selected"])
display(holdout_compare)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

sns.barplot(
    data=outer_summary_df,
    x="model_name",
    y="rmse_mean",
    ax=axes[0],
    palette="deep",
)
axes[0].set_title("Outer CV RMSE by Model")
axes[0].set_xlabel("model_name")
axes[0].set_ylabel("rmse_mean")

sns.barplot(
    data=holdout_compare,
    x="benchmark_label",
    y="rmse",
    ax=axes[1],
    palette="muted",
)
axes[1].set_title(f"{HOLDOUT_DISPLAY_TITLE} RMSE: Final Model vs Benchmarks")
axes[1].set_xlabel("benchmark_label")
axes[1].set_ylabel("rmse")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

In [ ]:
takeaway_rows = []
for _, selected_row in selected_model_df.iterrows():
    variant_key = selected_row["variant_key"]
    variant_compare = holdout_compare[holdout_compare["variant_key"] == variant_key].copy()
    best_benchmark = (
        variant_compare[variant_compare["benchmark_key"] != "selected_final_model"]
        .sort_values("delta_rmse_vs_selected")
        .iloc[0]
    )
    takeaway_rows.append(
        {
            "variant_key": variant_key,
            "selected_model_name": selected_row["model_name"],
            "selected_spec_name": selected_row["spec_name"],
            "outer_rmse_mean": float(selected_row["rmse_mean"]),
            "best_benchmark_label": best_benchmark["benchmark_label"],
            "holdout_delta_rmse_best_benchmark_vs_selected": float(best_benchmark["delta_rmse_vs_selected"]),
            "holdout_delta_oos_r2_best_benchmark_vs_selected": float(best_benchmark["delta_oos_r2_vs_selected"]),
        }
    )
takeaway_df = pd.DataFrame(takeaway_rows).sort_values(["variant_key", "holdout_delta_rmse_best_benchmark_vs_selected"])
display(takeaway_df)

In [ ]:
outer_splits_df.to_csv(RESULTS_DIR / "stage2_nested_cv_model_compare_outer_splits.csv", index=False)
best_params_df.to_csv(RESULTS_DIR / "stage2_nested_cv_model_compare_best_params_by_fold.csv", index=False)
outer_fold_metrics_df.to_csv(RESULTS_DIR / "stage2_nested_cv_model_compare_outer_fold_metrics.csv", index=False)
outer_summary_df.to_csv(RESULTS_DIR / "stage2_nested_cv_model_compare_outer_summary_metrics.csv", index=False)
outer_predictions_df.to_csv(RESULTS_DIR / "stage2_nested_cv_model_compare_outer_predictions.csv", index=False)
holdout_metrics_df.to_csv(RESULTS_DIR / "stage2_nested_cv_model_compare_holdout_metrics.csv", index=False)
holdout_predictions_df.to_csv(RESULTS_DIR / "stage2_nested_cv_model_compare_holdout_predictions.csv", index=False)
holdout_importances_df.to_csv(RESULTS_DIR / "stage2_nested_cv_model_compare_holdout_importances.csv", index=False)
selected_model_df.to_csv(RESULTS_DIR / "stage2_nested_cv_model_compare_selected_model.csv", index=False)
holdout_benchmark_metrics_df.to_csv(RESULTS_DIR / "stage2_nested_cv_model_compare_holdout_benchmark_metrics.csv", index=False)
holdout_benchmark_predictions_df.to_csv(RESULTS_DIR / "stage2_nested_cv_model_compare_holdout_benchmark_predictions.csv", index=False)
holdout_compare.to_csv(RESULTS_DIR / "stage2_nested_cv_model_compare_holdout_compare.csv", index=False)

print("Saved model-compare artifacts:")
for path in sorted(RESULTS_DIR.glob("stage2_nested_cv_model_compare_*.csv")):
    print(path)